## 0️⃣ 환경 설정

In [ ]:
from dotenv import load_dotenv
# 같은 위치의 .env 파일을 메모리에 로드하는 함수
load_dotenv(override=True)

In [ ]:
#필요한 라이브러리 설치
# !pip install langchain_text_splitters
# !pip install langchain_community
# !pip install langchain_openai
# !pip install pymupdf
# !pip install faiss-cpu 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

## 1️⃣ 로드 (Load)

In [ ]:
# Step 1 : 로드
loader = PyMuPDFLoader("document/RAG_test_report.pdf")
docs = loader.load()

# 해당 로더는 기본적으로 페이지단위로 분할을 함
print(f"문서의 페이지수: {len(docs)}")

## 2️⃣ 문서분할 (Split)

In [ ]:
# 단계 2: 문서 분할(Split Documents)

# 청크사이즈 및 오버렙 사이즈 정의
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

#실제로 split_documents 를 이용해 문서를 자름
split_documents = text_splitter.split_documents(docs)

print(f"분할된 청크의수: {len(split_documents)}")

## 3️⃣ 임베딩 (Embed)

In [ ]:
# 단계 3: OpenAI 임베딩 로드
embeddings = OpenAIEmbeddings()

## 4️⃣ 벡터저장 (Vector Store)

In [ ]:
# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다. (로컬에 아직 저장 X)
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

-----------------

⬆️⬆️⬆️⬆️⬆️⬆️ 여기까지가 백터에 저장

----------------------------------------------

## 5️⃣ Retriever(검색기) 생성

In [ ]:

# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever(
    search_type="similarity",  # 기본값similarity, mmr 도있음
    search_kwargs={"k": 3}     
)

In [ ]:
# 검색기에서 직접 사용가능 관련성 높은것부터 내림차순
retriever.invoke("고영향 ' AI(high-impact AI)이 뭐야")

## 6️⃣프롬프트 생성

In [ ]:


# 프롬프트를 생성합니다.
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
                """질문 응답 작업을 위한 어시스턴트입니다.
                    다음에 제공되는 검색된(retrived) 컨텍스트를 사용하여 질문에 답하십시오.
                    답을 모르면 모른다고 말하십시오.
                    한국어로 답변하십시오."""
    ),
    (
        "human",
                """#Context:
                {context}

                #Question:
                {question}

                #Answer:"""
    )
])

## 7️⃣ LLM 생성

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 8️⃣ Chain 생성

In [ ]:

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 생성된 체인에 쿼리(질문)을 입력하고 실행합니다.
question = "CAIO가 뭐야?"



#Stream 형식으로 답변
for chunk in chain.stream(question):
    print(chunk, end="", flush=True)


## 9️⃣ 전체 코드

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Step 1 : 로드
loader = PyMuPDFLoader("document/RAG_test_report.pdf")
docs = loader.load()

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings()

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
                """질문 응답 작업을 위한 어시스턴트입니다.
                    다음에 제공되는 검색된(retrived) 컨텍스트를 사용하여 질문에 답하십시오.
                    답을 모르면 모른다고 말하십시오.
                    한국어로 답변하십시오."""
    ),
    (
        "human",
                """#Context:
                {context}

                #Question:
                {question}

                #Answer:"""
    )
])

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 생성된 체인에 쿼리(질문)을 입력하고 실행합니다.
question = "CAIO가 뭐야?"



#Stream 형식으로 답변
for chunk in chain.stream(question):
    print(chunk, end="", flush=True)
